# S0c - plots and arrays

## Learning objectives

By the end of this notebook you can:

1. draw a line, a histogram and a scatter plot, with axes that say what they show
2. read a figure critically: what it shows, and what it hides
3. explain what a histogram tells you that an average does not
4. use a NumPy array and say why it is not a list
5. compute the mean and standard deviation of an array

## Prerequisites

- `S0a` (lists, loops, functions) and `S0b` (loading data, selecting columns).
- The data pack is built (`python build/prepare_data.py --auto`).

**Time:** about 1 hour

## Engineering context

Two AEOLIS datasets, both of which come back in later sessions. The factory machine log you met in `S0b`, and the site's hourly electricity consumption - four years of it, which you will be asked to forecast in Session 4.

A four-year series is 34,589 numbers. You cannot read that. You can look at it.

## Why this matters

Over six sessions you will read perhaps eighty figures and write almost none - the course provides the plotting code so that your lab time goes on interpretation instead of on `plt.xlabel`.

But you cannot interpret what you cannot read. And there is a harder reason: **a single number can hide almost anything.** Two datasets can share a mean, a standard deviation and a correlation, and look nothing alike. The only reliable defence is to look at the data, which means being fluent enough with a plot to trust what you see.

In [ ]:
# Run me first. Sets the course up on Colab; does nothing if you are already local.
COURSE_REPO = ""
DRIVE_FOLDER = "AEOLIS_course"


def _setup():
    import importlib, os, subprocess, sys
    from pathlib import Path

    try:
        importlib.import_module("aeolis")
        return "already available"
    except ModuleNotFoundError:
        pass

    for candidate in [Path(f"/content/drive/MyDrive/{DRIVE_FOLDER}"), Path("/content/course")]:
        if (candidate / "aeolis").is_dir():
            os.chdir(candidate)
            sys.path.insert(0, str(candidate))
            return f"found at {candidate}"

    if COURSE_REPO:
        target = Path("/content/course")
        subprocess.run(["git", "clone", "--depth", "1", COURSE_REPO, str(target)], check=True)
        os.chdir(target)
        sys.path.insert(0, str(target))
        return "cloned from the course repository"

    raise SystemExit(
        "The course files are not here yet. Do ONE of these, then re-run this cell:\n\n"
        "  1. Ask your instructor for the repository link and run, in a new cell:\n"
        "       !git clone <link> /content/course\n\n"
        f"  2. Put the course folder in your Google Drive as '{DRIVE_FOLDER}', then run:\n"
        "       from google.colab import drive; drive.mount('/content/drive')\n\n"
        "  3. Run this notebook on your own computer, inside the course folder."
    )


print("course:", _setup())

import aeolis
from pathlib import Path

needed = ["data/ai4i.csv", "data/consumption.parquet"]
missing = [f for f in needed if not Path(f).exists()]
if missing:
    print("data pack: MISSING", missing)
    print("           the instructor runs: python build/prepare_data.py --auto")
else:
    print("data pack: ok")
print("ready")

## Code Companion

Every exercise below can be solved by adapting one of these. Copy, change the names, run. That is not cheating - it is how data code actually gets written.

```python
# the one import you need
import matplotlib.pyplot as plt

# a line - use it for anything over time
plt.plot(x, y)

# a distribution - how often each value occurs
plt.hist(values, bins=50)

# a relationship between two measurements
plt.scatter(x, y)

# ALWAYS label - an unlabelled axis is not a result
plt.xlabel('time'); plt.ylabel('kW')

# say what the figure is
plt.title('Site consumption')

# make it big enough to read
plt.figure(figsize=(11, 3))

# show it
plt.show()

# a NumPy array from a list
import numpy as np; a = np.array([1.0, 2.0, 3.0])

# arithmetic on every element at once
b = a * 2

# statistics
a.mean(), a.std(), a.min(), a.max()

# a regular sequence
np.arange(0, 10)

# how many, and what shape
a.shape
```

## 1. A line: anything that happens over time

Load the consumption series and look at one week of it. One week, not four years - choosing the window is part of the skill.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

consumption = pd.read_parquet('data/consumption.parquet')
week = consumption['consumption_kw'].iloc[2000:2168]   # 168 hours = 7 days

plt.figure(figsize=(11, 3))
plt.plot(week.index, week.values)
plt.xlabel('date')
plt.ylabel('consumption (kW)')
plt.title('AEOLIS site consumption - one week')
plt.show()

**Look before you read on.** There is structure in that line, and you can describe it without any statistics:

- it rises and falls once a day - people arrive, people leave
- the peaks are not all the same height
- some days look different from others

Those three sentences are a **decomposition**, which is the formal name Session 4 gives it: a daily cycle, plus a slower drift, plus noise. You have just done by eye what the first half of Session 4 does with code.

Now the same data, all four years of it. Predict first: **what will the yearly pattern look like, and why?**

In [ ]:
daily = consumption['consumption_kw'].resample('D').mean()

plt.figure(figsize=(11, 3))
plt.plot(daily.index, daily.values, linewidth=0.7)
plt.xlabel('date')
plt.ylabel('mean daily consumption (kW)')
plt.title('Four years, averaged to one point per day')
plt.show()

Winter high, summer low, four times over - heating. Two things to take from this:

1. **The pattern was invisible at one-week zoom and obvious at four-year zoom.** The window you choose determines what you can see. Neither view is the correct one.
2. `resample('D').mean()` collapsed 24 hourly values into one daily value. That made the yearly cycle visible and **destroyed** the daily cycle. Every summary throws something away; the skill is knowing what.

**Your turn.** Plot the first 48 hours - two days - of the series.

In [ ]:
two_days = consumption['consumption_kw'].iloc[0:48]

plt.figure(figsize=(11, 3))
plt.plot(...)                      # TODO (1): the index on x, the values on y
plt.xlabel(...)                    # TODO (2): what is on the x axis?
plt.ylabel(...)                    # TODO (3): and the y axis? include the unit
plt.title('Two days')
plt.show()

**Hint 1.** Copy the four lines from the one-week cell above and change the variable name.

**Hint 2.** `plt.plot(two_days.index, two_days.values)`, then `'date'` and `'consumption (kW)'`.

## 2. A histogram: what an average cannot tell you

A histogram sorts the values into bins and counts how many land in each. It answers "what does a typical value look like, and how much do they vary?" - which an average cannot.

In [ ]:
torque = pd.read_csv('data/ai4i.csv')['Torque [Nm]']

plt.figure(figsize=(9, 3))
plt.hist(torque, bins=50)
plt.axvline(torque.mean(), color='red', linewidth=2, label=f'mean {torque.mean():.1f}')
plt.axvline(torque.median(), color='orange', linewidth=2, linestyle='--',
            label=f'median {torque.median():.1f}')
plt.xlabel('torque (Nm)')
plt.ylabel('number of runs')
plt.title('Torque across 10,000 production runs')
plt.legend()
plt.show()

A single hump, roughly symmetric, centred near 40 Nm, nearly all values between 20 and 60. Mean and median sit almost on top of each other - **that is what symmetric looks like.**

Now a column where they do not:

In [ ]:
speed = pd.read_csv('data/ai4i.csv')['Rotational speed [rpm]']

plt.figure(figsize=(9, 3))
plt.hist(speed, bins=60)
plt.axvline(speed.mean(), color='red', linewidth=2, label=f'mean {speed.mean():.0f}')
plt.axvline(speed.median(), color='orange', linewidth=2, linestyle='--',
            label=f'median {speed.median():.0f}')
plt.xlabel('rotational speed (rpm)')
plt.ylabel('number of runs')
plt.title('Rotational speed - note the long right tail')
plt.legend()
plt.show()

The bulk sits near 1500 rpm, and a thin tail stretches out past 2800. The mean (1539) is pulled to the right of the median (1503) by that tail.

**This is the reason to plot before you summarise.** "Average speed 1539 rpm" suggests a typical run at 1539. Very few runs are anywhere near it, and the handful of high-speed runs - which is precisely where some failures live - are invisible in the average. In Session 1 you will meet columns where the tail is not a tail at all, but a broken sensor writing the same wrong number.

In [ ]:
# TODO: a histogram of process temperature, with 40 bins and both axes labelled
process_temp = pd.read_csv('data/ai4i.csv')['Process temperature [K]']

plt.figure(figsize=(9, 3))
...
plt.show()

**Hint 1.** Three lines: `plt.hist(...)`, `plt.xlabel(...)`, `plt.ylabel(...)`. The unit is K.

## 3. A scatter plot: two measurements at once

One dot per run: torque across, speed up. Then colour the dots by whether the machine failed.

In [ ]:
df = pd.read_csv('data/ai4i.csv')
ok = df[df['Machine failure'] == 0]
bad = df[df['Machine failure'] == 1]

plt.figure(figsize=(7, 5))
plt.scatter(ok['Torque [Nm]'], ok['Rotational speed [rpm]'],
            s=4, alpha=0.25, label='ran fine')
plt.scatter(bad['Torque [Nm]'], bad['Rotational speed [rpm]'],
            s=14, color='red', label='failed')
plt.xlabel('torque (Nm)')
plt.ylabel('rotational speed (rpm)')
plt.title('10,000 runs: where do the failures live?')
plt.legend()
plt.show()

**Study that figure before reading on.** The red dots are not scattered at random. They gather in two regions: high torque with low speed, and low torque with high speed.

Torque multiplied by speed is roughly mechanical power. Both red regions are places where that product is extreme - too much load, or too little. So the plot is pointing at a physical mechanism, and it is suggesting a feature nobody handed you: **power**, the product of two columns you do have.

That move - inventing a column that captures the physics - is called **feature engineering**, and Session 1 spends thirty-five minutes on it. You have just found the reason it works.

Two craft details that made the figure readable: `alpha=0.25` makes the 9,661 blue dots translucent so the pile-up is visible instead of a solid block, and the red dots are drawn larger and second so 339 of them are not buried under 9,661. **Ten thousand dots at full opacity would have shown you nothing.**

## 4. NumPy arrays: the floor everything stands on

A NumPy array looks like a list of numbers and behaves very differently. Arithmetic applies to **every element at once**, with no loop.

In [ ]:
import numpy as np

as_list = [1.0, 2.0, 3.0, 4.0]
as_array = np.array([1.0, 2.0, 3.0, 4.0])

print('list  * 2 :', as_list * 2)      # repeats the list!
print('array * 2 :', as_array * 2)     # doubles each number

`list * 2` repeated the list; `array * 2` doubled every value. This is the single most common source of confusion when moving from plain Python to data code, and now you have seen it once, deliberately.

Arrays carry their statistics, and they know their own shape:

In [ ]:
temps = np.array([67.1, 68.4, 71.9, 74.2, 69.8, 68.0])

print('mean   :', temps.mean().round(2))
print('std dev:', temps.std().round(2))
print('shape  :', temps.shape)
print('above 70:', temps[temps > 70])       # the same filtering idea as pandas

**Standard deviation** in one sentence: the typical distance of a value from the mean. Small means the values huddle; large means they spread. It is the number you reach for when "the average is 70" is not enough.

`np.arange` builds a regular sequence, which is how you make an x axis when the data has no timestamps:

In [ ]:
hours = np.arange(0, 24)
print(hours)
print('shape:', hours.shape)

**Why you should care.** pandas is built on NumPy: `df['Torque [Nm]'].values` *is* an array. So is every image, every batch of text and every set of model weights in Session 3 - PyTorch tensors are the same idea with a GPU attached. When Session 3 says "the input has shape (32, 5)", it means a NumPy-shaped block of 32 rows and 5 columns. Recognising `.shape` is most of what you need to debug a neural network.

In [ ]:
readings = np.array([12.5, 14.1, 11.8, 15.9, 13.2, 22.7])

# TODO (1): the mean
mean_reading = ...

# TODO (2): the standard deviation
std_reading = ...

# TODO (3): only the readings above 15
large = ...

**Hint 1.** All three patterns are in the cell two above this one.

In [ ]:
from aeolis.checks import done, close
done(mean_reading, std_reading, large)
assert close(mean_reading, 15.033, tol=0.01), f'expected about 15.03, got {mean_reading}'
assert close(std_reading, 3.666, tol=0.01), f'expected about 3.67, got {std_reading}'
assert len(large) == 2, f'expected 2 readings above 15, got {len(large)}'
print('correct')

Notice something in your own answer: the mean is 15.03, and only **two** of the six readings are above it. One value of 22.7 dragged the average above two thirds of the data. Six numbers were enough to mislead an average - which is the section 2 lesson again, and the reason Session 1 opens by plotting before summarising.

## Knowledge check

Four tasks. The last one has no code at all - it is the kind of question this course actually assesses.

In [ ]:
df = pd.read_csv('data/ai4i.csv')

# 1. The mean air temperature, as a NumPy array operation
air = df['Air temperature [K]'].values
mean_air = ...

# 2. How many runs had tool wear above the mean tool wear?
wear = df['Tool wear [min]'].values
n_above_mean = ...

# 3. Draw a histogram of torque with 30 bins, both axes labelled
#    (no assertion for this one - it is checked by eye)

# 4. In one sentence: the mean rotational speed is 1539 rpm and the median is 1503 rpm.
#    What does that gap tell you, and what would you look at next?
answer_4 = '...'

In [ ]:
from aeolis.checks import done, close
done(mean_air, n_above_mean, answer_4)
score = 0
score += close(mean_air, 300.0, tol=0.2)
score += (n_above_mean == 5018)
score += (len(answer_4) > 60)          # a real sentence, not a placeholder
print(f'score: {score}/3 on the automatic checks')
print('ready for Session 1' if score >= 2 else 'review the sections you missed, then retry')

## Reflection

1. The four-year plot showed a yearly cycle that the one-week plot could not. Give one thing the four-year plot hides that the one-week plot showed.
2. In the scatter plot, the failures clustered at extreme combinations of torque and speed. What new column would you build from those two, and why?

In [ ]:
reflection = """
1.
2.
"""

## Summary

- `plt.plot` for time, `plt.hist` for a distribution, `plt.scatter` for a relationship.
- **Label the axes, with units.** An unlabelled axis is not a result.
- The window and the summary you choose decide what you can see - and what you destroy.
- Mean far from median means a tail, and a tail means the average is misleading someone.
- A NumPy array does arithmetic on every element; a list does not. `.shape` matters, and will matter more in Session 3.
- In the six sessions the plotting code is provided. Reading the figure is your job.

## Common mistakes

- **No `plt.show()`.** In some environments nothing appears until you call it. Make it a habit.
- **Unlabelled axes.** Marked down every time, in every session, deliberately.
- **10,000 opaque dots.** Use `alpha` and a small marker size, or the plot is a solid block.
- **`list * 2` when you meant an array.** It repeats the list instead of doubling the values.
- **Trusting a mean you have not plotted.** The whole point of section 2.

## Further exploration (optional)

- `plt.subplots(2, 1)` puts two panels in one figure - Session 4 uses it for decomposition.
- Anscombe's quartet: four datasets with identical means, variances and correlations that look completely different when plotted. Search for the picture; it takes ten seconds and it is the best argument for plotting ever made.
- `df.plot()` is a pandas shortcut for quick looks. Fine for exploring, too crude for a report.

---

## You are ready

Three notebooks: Python, pandas, plots and arrays. That is the whole toolkit the course assumes.

**Bring to Session 1:** a working environment (Colab or local), these three notebooks completed, and your knowledge-check scores. Session 1 opens with a twelve-minute entry check drawn from exactly these objectives - not to catch you out, but so that nobody spends the practical stuck on square brackets while the rest of the room is deciding what to predict.

In Session 1 the data stops being tidy.